In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import shuffle
# 导入 SMOTE 库
from imblearn.over_sampling import SMOTE 

# 1. 读取数据
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2. 预处理：处理 TotalCharges 隐藏空值并填充
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

# 3. 编码：标签转数值 & 移除 ID & One-Hot Encoding
if 'Churn' in df.columns:
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df.drop('customerID', axis=1, inplace=True)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 4. 准备过采样
# SMOTE 需要将特征 (X) 和标签 (y) 分开处理
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# 实例化 SMOTE，将少数类提升到与多数类数量一致
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# 将平衡后的数据合并回 DataFrame
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['Churn'] = y_resampled

# 5. Normalization：缩放数据
scaler = MinMaxScaler()
# 注意：fit_transform 会返回 numpy 数组，我们需要转回 DataFrame
df_normalized = pd.DataFrame(scaler.fit_transform(df_resampled), columns=df_resampled.columns)

# 6. Shuffle：打乱数据
# 经过 SMOTE 后的数据通常是按类别排列的，必须打乱
df_final = shuffle(df_normalized, random_state=42).reset_index(drop=True)

# 7. 输出结果
df_final.to_csv("CleanedDataset.csv", index=False)

print("处理成功并已完成过采样！")
print(f"原始数据形状: {df_encoded.shape}")
print(f"过采样后数据形状: {df_final.shape}")
print(f"各类别数量分布: \n{df_final['Churn'].value_counts()}")

处理成功并已完成过采样！
原始数据形状: (7043, 31)
过采样后数据形状: (10348, 31)
各类别数量分布: 
Churn
1.0    5174
0.0    5174
Name: count, dtype: int64
